In [87]:
import requests
import pandas as pd
import io
from pathlib import Path
import re

OUTPUT_DIR = Path.cwd()

url = "https://data.bs.ch/api/explore/v2.1/catalog/datasets/100189/exports/csv"
params = {"delimiter": ";"}

df = None
try:
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), sep=";")
    print(df.shape)
    print(df.head())
except Exception:
    print("Server im Basel down")

(1456, 11)
                            geo_point_2d  \
0   47.54681651130626, 7.610305589714874   
1   47.55151799271766, 7.595790368620413   
2  47.552615585349024, 7.593317647881711   
3   47.55740752860826, 7.573272830772866   
4   47.55245246358092, 7.601139099082942   

                                           geo_shape  id_strasse  \
0  {"coordinates": [[[7.609905, 47.54731], [7.610...           1   
1  {"coordinates": [[[7.595343, 47.551116], [7.59...           4   
2  {"coordinates": [[[7.592077, 47.553707], [7.59...           5   
3  {"coordinates": [[[7.574517, 47.557281], [7.57...           7   
4  {"coordinates": [[[7.598168, 47.55192], [7.599...           9   

       strassenname strassenname_kurz strassenindex gemeindename  \
0      Adlerstrasse         Adlerstr.           ADL        Basel   
1      Aeschenplatz      Aeschenplatz           AES        Basel   
2   Aeschenvorstadt   Aeschenvorstadt           AES        Basel   
3      Ahornstrasse         Ahornstr.      

In [88]:
def _norm(s: str) -> str:
    """Vereinheitlicht Spaltennamen für den Vergleich."""
    s = str(s).strip().lstrip("\ufeff").lower()
    s = s.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")
    return re.sub(r"[^a-z0-9]", "", s)

# Zielspalte -> akzeptierte Schreibweisen (der Zielname selbst zählt automatisch auch)
ALIASES = {
    "Id Strasse":             ["id_strasse", "id-strasse", "id strasse"],
    "Strassenname":           ["strassenname", "strasse"],
    "Erklärung erste Zeile":  ["erklaerung_erste_zeile"],
    "Erklärung zweite Zeile": ["erklaerung_zweite_zeile"],
    "Geo Shape":              ["geo_shape"],
    "Geo Point":              ["geo_point_2d", "geo_point"],
    "Erstmals erwähnt":       ["erstmalige_erwaehnung", "erstmals_erwaehnt"],
    "Amtlich benannt":        ["amtliche_benennung", "amtlich_benannt"],
    "Indextext":              ["strassenindex", "index"],
    "Kurztext":               ["strassenname_kurz", "kurz"],
    "Gemeindename":           ["Gemeindename", "gemeinde"],
}
ZIEL_SPALTEN = list(ALIASES.keys())

if df is not None:
    # Nachschlage-Tabelle: normalisierter Name -> Zielspalte
    lookup = {}
    for ziel, namen in ALIASES.items():
        for n in [ziel] + namen:
            lookup[_norm(n)] = ziel

    neue_namen, unbekannt = {}, []
    for col in df.columns:
        key = _norm(col)
        if key in lookup:
            neue_namen[col] = lookup[key]
        else:
            unbekannt.append(col)

    df = df.rename(columns=neue_namen)

    # --- Warnungen, damit du Änderungen von Basel sofort bemerkst ---
    fehlend = [z for z in ZIEL_SPALTEN if z not in set(neue_namen.values())]
    if fehlend:
        print("WARNUNG: Diese Zielspalten wurden NICHT gefunden und bleiben LEER:", fehlend)
    if unbekannt:
        print("HINWEIS: Diese Quellspalten ist unbekannt und wird leer gelassen:", unbekannt)
    elif unbekannt:
        print("Alle Kategorien wurden erkannt.")

    df = df.reindex(columns=ZIEL_SPALTEN)

In [89]:
# Auf Basel filtern – Spalte 'Gemeindename' robust finden
gem_col = next((c for c in df.columns if _norm(c) in ("gemeindename", "gemeinde")), None)
if gem_col is None:
    raise SystemExit("Spalte 'Gemeindename' nicht gefunden!")
df = df[df[gem_col].fillna("").str.strip().str.casefold() == "basel"].copy()

In [90]:
# CSV speichern – ersetzt vorhandene Datei automatisch
if df is not None:
    output_path = OUTPUT_DIR / "Strassen_export_01.csv"
    try:
        df.to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")
        print(f"Gespeichert unter: {output_path}")
    except PermissionError:
        print("Datei noch geöffnet -> Datei noch schliessen erst.")
else:
    print("Nichts zu speichern.")

Gespeichert unter: c:\Users\Jimmy\Documents\00_Jimmydok\06_Schule\01_FHNW\01_Kursunterlagen\4.Semester\4040-Geoprogrammierung_II\Hackaton\Projekte\StreetLore-Basel\Strassen_export_01.csv
